## 三端静态输出核对

In [77]:
import time
from pathlib import Path
import pandas as pd

from fefetlab.instruments.visa_session import VisaConfig, VisaSession
from fefetlab.b1500 import B1500, B1500Error


cfg = VisaConfig(
    resource="GPIB0::17::INSTR",
    timeout_ms=30000,
    write_termination="\r\n",
    read_termination="\r\n",
    send_end=True,
)

# 第一组候选映射：先按这个试
CH_G = 4
CH_D = 5
CH_S = 6

# 小信号 sanity 用
I_COMP = 1e-3
VRANGE_G = 0
VRANGE_D = 0
VRANGE_S = 0

print("候选映射：")
print(f"G = CH{CH_G}, D = CH{CH_D}, S = CH{CH_S}")


session = VisaSession(cfg)
session.open()
b = B1500(session)

print("IDN:", session.query("*IDN?"))
print("ERRX:", b.errx())

def safe_zero_and_cl(channels):
    try:
        b.dz(channels)
    except Exception as e:
        print("DZ warning:", e)

    try:
        b.cl(channels)
    except Exception as e:
        print("CL warning:", e)

候选映射：
G = CH4, D = CH5, S = CH6
IDN: Agilent Technologies,B1500A,MY55231213,A.06.02.2023.0401
ERRX: +0,"No Error."


In [82]:
static_sets = [
    {"name": "set1", "vg": 0.0,  "vd": 0.0,  "vs": 0.0},
    {"name": "set2", "vg": -0.1, "vd": 0.05, "vs": 0.1},
    {"name": "set3", "vg": -0.2, "vd": 0.10, "vs": 0.3},
    {"name": "set4", "vg": 0.0,  "vd": 0.10, "vs": 0.5},
]
pd.DataFrame(static_sets)



,name,vg,vd,vs
0,set1,0.0,0.00,0.0
1,set2,-0.1,0.05,0.1
2,set3,-0.2,0.10,0.3
3,set4,0.0,0.10,0.5


### 单组静态输出

In [83]:
s = static_sets[0]

print(f"\n===== {s['name']} =====")
print(f"设定: Vg={s['vg']} V, Vd={s['vd']} V, Vs={s['vs']} V")

b.cn([CH_G, CH_D, CH_S])
b.dv(CH_G, VRANGE_G, s["vg"], I_COMP)
b.dv(CH_D, VRANGE_D, s["vd"], I_COMP)
b.dv(CH_S, VRANGE_S, s["vs"], I_COMP)

time.sleep(0.5)
print("现在看示波器/万用表")


===== set1 =====
设定: Vg=0.0 V, Vd=0.0 V, Vs=0.0 V
现在看示波器/万用表


In [84]:
s = static_sets[1]

print(f"\n===== {s['name']} =====")
print(f"设定: Vg={s['vg']} V, Vd={s['vd']} V, Vs={s['vs']} V")

b.cn([CH_G, CH_D, CH_S])
b.dv(CH_G, VRANGE_G, s["vg"], I_COMP)
b.dv(CH_D, VRANGE_D, s["vd"], I_COMP)
b.dv(CH_S, VRANGE_S, s["vs"], I_COMP)

time.sleep(0.5)
print("现在看示波器/万用表")


===== set2 =====
设定: Vg=-0.1 V, Vd=0.05 V, Vs=0.1 V
现在看示波器/万用表


In [85]:
s = static_sets[2]

print(f"\n===== {s['name']} =====")
print(f"设定: Vg={s['vg']} V, Vd={s['vd']} V, Vs={s['vs']} V")

b.cn([CH_G, CH_D, CH_S])
b.dv(CH_G, VRANGE_G, s["vg"], I_COMP)
b.dv(CH_D, VRANGE_D, s["vd"], I_COMP)
b.dv(CH_S, VRANGE_S, s["vs"], I_COMP)

time.sleep(0.5)
print("现在看示波器/万用表")


===== set3 =====
设定: Vg=-0.2 V, Vd=0.1 V, Vs=0.3 V
现在看示波器/万用表


In [81]:
s = static_sets[3]

print(f"\n===== {s['name']} =====")
print(f"设定: Vg={s['vg']} V, Vd={s['vd']} V, Vs={s['vs']} V")

b.cn([CH_G, CH_D, CH_S])
b.dv(CH_G, VRANGE_G, s["vg"], I_COMP)
b.dv(CH_D, VRANGE_D, s["vd"], I_COMP)
b.dv(CH_S, VRANGE_S, s["vs"], I_COMP)

time.sleep(0.5)
print("现在看示波器/万用表")


===== set4 =====
设定: Vg=0.0 V, Vd=0.1 V, Vs=0.0 V
现在看示波器/万用表


In [23]:
import inspect
from fefetlab.b1500 import B1500

print(inspect.signature(B1500.dv))

(self, ch: 'int', a: 'float | int', b: 'float', c: 'float | int' = 0) -> 'None'


SMU1是G
SMU2是D
SMU3是S

### 步进

In [57]:
step_seq_gate = [0.0, -0.1, -0.2, -0.5, -1.0, -0.5, -0.2, -0.1, 0.0]
step_seq_drain = [0.0, 0.05, 0.1, 0.2, 0.1, 0.05, 0.0]

print("Gate step:", step_seq_gate)
print("Drain step:", step_seq_drain)

Gate step: [0.0, -0.1, -0.2, -0.5, -1.0, -0.5, -0.2, -0.1, 0.0]
Drain step: [0.0, 0.05, 0.1, 0.2, 0.1, 0.05, 0.0]


In [63]:
hold_s = 2.0   # 每一步保持 2 秒，方便看示波器

for vg in step_seq_gate:
    print(f"\n===== Gate step: Vg = {vg} V =====")

    b.cn([CH_G])
    b.dv(CH_G, VRANGE_G, vg, I_COMP)

    print(f"请现在看示波器/万用表：G 端应约为 {vg} V")
    time.sleep(hold_s)

safe_zero_and_cl([CH_G])
print("Gate step 核对完成，已归零。")


===== Gate step: Vg = 0.0 V =====
请现在看示波器/万用表：G 端应约为 0.0 V

===== Gate step: Vg = -0.1 V =====
请现在看示波器/万用表：G 端应约为 -0.1 V

===== Gate step: Vg = -0.2 V =====
请现在看示波器/万用表：G 端应约为 -0.2 V

===== Gate step: Vg = -0.5 V =====
请现在看示波器/万用表：G 端应约为 -0.5 V

===== Gate step: Vg = -1.0 V =====
请现在看示波器/万用表：G 端应约为 -1.0 V

===== Gate step: Vg = -0.5 V =====
请现在看示波器/万用表：G 端应约为 -0.5 V

===== Gate step: Vg = -0.2 V =====
请现在看示波器/万用表：G 端应约为 -0.2 V

===== Gate step: Vg = -0.1 V =====
请现在看示波器/万用表：G 端应约为 -0.1 V

===== Gate step: Vg = 0.0 V =====
请现在看示波器/万用表：G 端应约为 0.0 V
Gate step 核对完成，已归零。


In [31]:
hold_s = 2.0

for vd in step_seq_drain:
    print(f"\n===== Drain step: Vd = {vd} V =====")

    b.cn([CH_D])
    b.dv(CH_D, VRANGE_D, vd, I_COMP)

    print(f"请现在看示波器/万用表：D 端应约为 {vd} V")
    time.sleep(hold_s)

safe_zero_and_cl([CH_D])
print("Drain step 核对完成，已归零。")


===== Drain step: Vd = 0.0 V =====
请现在看示波器/万用表：D 端应约为 0.0 V

===== Drain step: Vd = 0.05 V =====
请现在看示波器/万用表：D 端应约为 0.05 V

===== Drain step: Vd = 0.1 V =====
请现在看示波器/万用表：D 端应约为 0.1 V

===== Drain step: Vd = 0.2 V =====
请现在看示波器/万用表：D 端应约为 0.2 V

===== Drain step: Vd = 0.1 V =====
请现在看示波器/万用表：D 端应约为 0.1 V

===== Drain step: Vd = 0.05 V =====
请现在看示波器/万用表：D 端应约为 0.05 V

===== Drain step: Vd = 0.0 V =====
请现在看示波器/万用表：D 端应约为 0.0 V
Drain step 核对完成，已归零。


In [36]:
hold_s = 2.0
fixed_vd = 0.1
fixed_vs = 0.0

for vg in step_seq_gate:
    print(f"\n===== 3-terminal step =====")
    print(f"Vg = {vg} V, Vd = {fixed_vd} V, Vs = {fixed_vs} V")

    b.cn([CH_G, CH_D, CH_S])

    b.dv(CH_G, VRANGE_G, vg, I_COMP)
    b.dv(CH_D, VRANGE_D, fixed_vd, I_COMP)
    b.dv(CH_S, VRANGE_S, fixed_vs, I_COMP)

    print("请现在看示波器/万用表：")
    print(f"G 端应约为 {vg} V")
    print(f"D 端应约为 {fixed_vd} V")
    print(f"S 端应约为 {fixed_vs} V")

    time.sleep(hold_s)

safe_zero_and_cl([CH_G, CH_D, CH_S])
print("三端联动 step 核对完成，已归零。")


===== 3-terminal step =====
Vg = 0.0 V, Vd = 0.1 V, Vs = 0.0 V
请现在看示波器/万用表：
G 端应约为 0.0 V
D 端应约为 0.1 V
S 端应约为 0.0 V

===== 3-terminal step =====
Vg = -0.1 V, Vd = 0.1 V, Vs = 0.0 V
请现在看示波器/万用表：
G 端应约为 -0.1 V
D 端应约为 0.1 V
S 端应约为 0.0 V

===== 3-terminal step =====
Vg = -0.2 V, Vd = 0.1 V, Vs = 0.0 V
请现在看示波器/万用表：
G 端应约为 -0.2 V
D 端应约为 0.1 V
S 端应约为 0.0 V

===== 3-terminal step =====
Vg = -0.5 V, Vd = 0.1 V, Vs = 0.0 V
请现在看示波器/万用表：
G 端应约为 -0.5 V
D 端应约为 0.1 V
S 端应约为 0.0 V

===== 3-terminal step =====
Vg = -1.0 V, Vd = 0.1 V, Vs = 0.0 V
请现在看示波器/万用表：
G 端应约为 -1.0 V
D 端应约为 0.1 V
S 端应约为 0.0 V

===== 3-terminal step =====
Vg = -0.5 V, Vd = 0.1 V, Vs = 0.0 V
请现在看示波器/万用表：
G 端应约为 -0.5 V
D 端应约为 0.1 V
S 端应约为 0.0 V

===== 3-terminal step =====
Vg = -0.2 V, Vd = 0.1 V, Vs = 0.0 V
请现在看示波器/万用表：
G 端应约为 -0.2 V
D 端应约为 0.1 V
S 端应约为 0.0 V

===== 3-terminal step =====
Vg = -0.1 V, Vd = 0.1 V, Vs = 0.0 V
请现在看示波器/万用表：
G 端应约为 -0.1 V
D 端应约为 0.1 V
S 端应约为 0.0 V

===== 3-terminal step =====
Vg = 0.0 V, Vd = 0.1 V, Vs = 

### 三端点测

In [ ]:
# 当前已确认映射     确认映射
CH_G = 4
CH_D = 5
CH_S = 6
CH_X = 7   # 额外通道，当前不用

I_COMP = 1e-3
VRANGE_G = 0
VRANGE_D = 0
VRANGE_S = 0

# 第一轮三端点测：先少量离散点
vg_points = [0.0, -0.2, -0.5, -0.8]
vd_fixed = 0.1
vs_fixed = 0.0

print("当前三端映射：")
print(f"G = CH{CH_G}, D = CH{CH_D}, S = CH{CH_S}, X = CH{CH_X}")
print("Vg points =", vg_points)
print("Vd fixed  =", vd_fixed)
print("Vs fixed  =", vs_fixed)

In [ ]:
# 安全归零函数
def safe_zero_and_cl(channels):
    try:
        b.dz(channels)
    except Exception as e:
        print("DZ warning:", e)

    try:
        b.cl(channels)
    except Exception as e:
        print("CL warning:", e)

In [ ]:
# 三端单点测量函数
import time

def measure_3terminal_point(b, vg, vd=0.1, vs=0.0, delay_s=0.2):
    row = {
        "vg_set": vg,
        "vd_set": vd,
        "vs_set": vs,
        "ig_raw": None,
        "id_raw": None,
        "is_raw": None,
        "ig_A": None,
        "id_A": None,
        "is_A": None,
        "err": None,
        "status": "ok",
    }

    try:
        # 推荐的基础设置
        b.fmt(1)
        b.av(10, 1)
        b.fl(0)

        b.cn([CH_G, CH_D, CH_S])

        # 加三端偏压
        b.dv(CH_G, VRANGE_G, vg, I_COMP)
        b.dv(CH_D, VRANGE_D, vd, I_COMP)
        b.dv(CH_S, VRANGE_S, vs, I_COMP)

        time.sleep(delay_s)

        # 同时保留 raw 和解析值
        ig_raw = b._query(f"TI {CH_G},0", check_err=False)
        id_raw = b._query(f"TI {CH_D},0", check_err=False)
        is_raw = b._query(f"TI {CH_S},0", check_err=False)

        row["ig_raw"] = ig_raw
        row["id_raw"] = id_raw
        row["is_raw"] = is_raw

        row["ig_A"] = b._parse_scalar_response(ig_raw)
        row["id_A"] = b._parse_scalar_response(id_raw)
        row["is_A"] = b._parse_scalar_response(is_raw)

        row["err"] = b.errx()

    except Exception as e:
        row["status"] = "invalid"
        row["err"] = repr(e)

    finally:
        safe_zero_and_cl([CH_G, CH_D, CH_S])

    return row

In [ ]:
# 执行三端点测量，并收集结果
import pandas as pd

rows = []

for vg in vg_points:
    print(f"\n===== 点测: Vg={vg} V, Vd={vd_fixed} V, Vs={vs_fixed} V =====")
    r = measure_3terminal_point(
        b,
        vg=vg,
        vd=vd_fixed,
        vs=vs_fixed,
        delay_s=0.2,
    )
    rows.append(r)

    print("ig_raw =", r["ig_raw"])
    print("id_raw =", r["id_raw"])
    print("is_raw =", r["is_raw"])
    print("ig_A   =", r["ig_A"])
    print("id_A   =", r["id_A"])
    print("is_A   =", r["is_A"])
    print("err    =", r["err"])
    print("status =", r["status"])

df_spot = pd.DataFrame(rows)


df_spot

In [ ]:
# 简单先看 Id 是否随 Vg 有系统变化
print("按 Vg 排序后的结果：")
display(df_spot[["vg_set", "id_A", "ig_A", "is_A", "status", "err"]].sort_values("vg_set"))

from pathlib import Path
import time

run_dir = Path("runs") / time.strftime("%Y%m%d_%H%M%S_3terminal_spot")
run_dir.mkdir(parents=True, exist_ok=True)

df_spot.to_csv(run_dir / "spot_results.csv", index=False, encoding="utf-8-sig")
df_spot.to_json(run_dir / "spot_results.json", orient="records", force_ascii=False, indent=2)

print("结果保存到:", run_dir)